 ## preparation of the decomposed automaton

In [47]:
# generate n chains of length m, where n and m are given by the user, default n = 5, m = 10
# fix some order of the chains
# add dependencies on the edges such that the edge of one chain depends on the states of later chains (not the previous ones in the fixed order)
# the number of states on which a given edge depends is chosen uniformly from {0,1, 2, 3}..if there are multiple states, they need to be from different chains.

import networkx as nx
import matplotlib.pyplot as plt
from collections import defaultdict
import random

def gen_chains(n=5, m=10):
    chains = defaultdict(list)

    for i in range(n):
        G = nx.Graph()
        nodes = [j for j in range(m)]
        edges = list(zip(nodes[:-1], nodes[1:]))

        G.add_edges_from(edges)
        chains[i] = G

    return chains

def add_enabling(chains):
    n = len(chains)
    
    for chain_id in range(n-1):
        G = chains[chain_id]
        
        for edge in G.edges():
            if random.random() < 0.5:
                num_deps = random.randint(0, 3)
                
                if num_deps > 0:
                    later_chains = list(range(chain_id + 1, n))
                    selected_chains = random.sample(later_chains, min(num_deps, len(later_chains)))
                    
                    enabling_conditions = []
                    for enabling_chain in selected_chains:
                        enabling_G = chains[enabling_chain]
                        enabling_node = random.choice(list(enabling_G.nodes()))
                        enabling_conditions.append({'automata_id':enabling_chain,'node_id':enabling_node})
                    
                    G.edges[edge]['enabling'] = enabling_conditions
    
    return chains

chains = add_enabling(gen_chains())

In [ ]:
for chain_id in range(len(chains)):
    print(f"\nChain {chain_id}:")
    G = chains[chain_id]
    for edge in G.edges(data=True):
        print(f"Edge {edge[0]} -> {edge[1]}: {edge[2]}")


Chain 0:
Edge 0 -> 1: {'enabling': [{'automata_id': 1, 'node_id': 1}, {'automata_id': 4, 'node_id': 0}, {'automata_id': 2, 'node_id': 3}]}
Edge 1 -> 2: {}
Edge 2 -> 3: {'enabling': [{'automata_id': 1, 'node_id': 4}, {'automata_id': 4, 'node_id': 3}, {'automata_id': 2, 'node_id': 3}]}
Edge 3 -> 4: {'enabling': [{'automata_id': 3, 'node_id': 5}, {'automata_id': 2, 'node_id': 1}, {'automata_id': 1, 'node_id': 9}]}
Edge 4 -> 5: {}
Edge 5 -> 6: {'enabling': [{'automata_id': 4, 'node_id': 8}, {'automata_id': 3, 'node_id': 4}]}
Edge 6 -> 7: {}
Edge 7 -> 8: {}
Edge 8 -> 9: {'enabling': [{'automata_id': 3, 'node_id': 1}, {'automata_id': 1, 'node_id': 3}]}

Chain 1:
Edge 0 -> 1: {}
Edge 1 -> 2: {}
Edge 2 -> 3: {}
Edge 3 -> 4: {'enabling': [{'automata_id': 3, 'node_id': 5}]}
Edge 4 -> 5: {'enabling': [{'automata_id': 3, 'node_id': 8}, {'automata_id': 4, 'node_id': 9}]}
Edge 5 -> 6: {'enabling': [{'automata_id': 4, 'node_id': 5}, {'automata_id': 3, 'node_id': 5}]}
Edge 6 -> 7: {'enabling': [{'aut

In [59]:
chains[0].nodes

NodeView((0, 1, 2, 3, 4, 5, 6, 7, 8, 9))

## creation of 1 sample problem

In [49]:
# for the fixed order of chains, create a vector of states for each chain. states in different chains are chosen randomly.
# choose 1 chain as the target chain
# choose a random target state in the target chain
# the result is sample = {"input": vector of states, "target": target state, "target chain": target chain}

In [50]:
from itertools import product
def get_vectors(chains):
    states_per_chain = []
    for chain_id in range(len(chains)):
        states_per_chain.append(list(chains[chain_id].nodes()))
    
    return list(product(*states_per_chain))

vecs = get_vectors(chains)

In [51]:
from tqdm import tqdm
id_2_vec = {}
vec_2_id = {}
for i, comb in tqdm(enumerate(vecs)):
    id_2_vec[i] = comb
    vec_2_id[comb] = i

0it [00:00, ?it/s]

100000it [00:00, 2157319.64it/s]


In [204]:
def get_sample(vector, chains):
    num_chains = len(chains)
    num_states = len(chains[0].nodes())
    
    target_chain = random.randint(0, num_chains-1)

    target_state = random.randint(0, num_states-1)
    
    return {
        'input': list(vector),
        'target_state': target_state,
        'target_chain': target_chain
    }

In [225]:
sample = get_sample(vecs[12223], chains)
sample

{'input': [1, 2, 2, 2, 3], 'target_state': 8, 'target_chain': 1}

## implementation of the solver

In [ ]:
def get_next(target_state, current_state, target_chain):
    if target_state < current_state[target_chain]:
        return current_state[target_chain] - 1
    else:
        return current_state[target_chain] + 1

def get_rule(chains, target_chain, current_state, target_state):
    return chains[target_chain].get_edge_data(current_state[target_chain], get_next(target_state, current_state, target_chain))

def solve(sample, chains, path=[]):
    target_chain = sample["target_chain"]
    target_state = sample["target_state"]
    current_state = list(sample["input"])
    # for the target chain, find all shortest paths from the current state to the target state
    nodes_rule = get_rule(chains, target_chain, current_state, target_state)
    # edge = chains[current_state[target_chain]].edges(current_state[target_chain])
    # print(edge)
    if nodes_rule == {}:
        next = get_next(target_state, current_state, target_chain)
        path.append((current_state[target_chain], next))
        print(path)
        sample["input"][target_chain] = next
        return solve(sample, chains, path)
    else:
        enabling = get_rule(chains, target_chain, current_state, target_state)
        next = get_next(target_state, current_state, target_chain)
        sample["input"][target_chain] = next
        sample["target_chain"] = enabling[0]["automata_id"]

    # take the first path and try to traverse it. ie iterate over the edges and do the following:
    # # if the edge is not dependent on other states:
    # # # cross the edge, i.e. change the current state for the target chain to get one more step closer to the target state
    # # # path.append(edge)
    # # else get the dependency states, depenendet = edge['enabling']
    # # for each dependency state, do the following:
    # # # new_sample = {}
    # # # new_sample['input_states'] = current_state
    # # # new_sample["target"] = dependency_state
    # # # new_sample['target chain'] = chain of the dependency state
    # # # path, current_state = solve(new_sample,path)
    # # after each dependency states are resolved, cross the edge and append it to the path and change the current state
    # after the shortest path to the target state if traversed, return the path and the current state

In [229]:
solve(sample, chains)

{'enabling': [{'automata_id': 3, 'node_id': 5}]}
